# Smart Waste Collection Agent - IDA* Algorithm
**Assignment 1 – PS1**

---

## Problem

We have a Smart Waste Collection Robot operating in Bengaluru city. The city's road network is represented as a weighted graph — nodes are waste collection centers / processing plants, and edges are roads with associated travel costs.

The robot needs to find the **cheapest route** from a given source to a destination. Since it runs on limited battery, we want to minimize the total travel cost.

We'll use **IDA*** (Iterative Deepening A*) to solve this.

---

## 1. PEAS Analysis

| Component | Description |
|-----------|-------------|
| **Performance** | Minimize total travel cost from source to destination; find the optimal (cheapest) path |
| **Environment** | Weighted undirected graph — static, fully observable, deterministic. Nodes are locations, edges are roads with costs |
| **Actuators** | Move along a road from current node to a neighboring node |
| **Sensors** | Knows current location, connected roads + their weights, and where the goal is |

**Agent type**: Goal-based, operating in a fully observable, deterministic, static, discrete environment.

## 2. Heuristic — Explanation and Justification

**hop-count based heuristic**:

**Heuristic Function:**
$$h(n) = hops(n, goal) \times d_{min}$$

```
h(n) = min_hops(n, goal) × min_edge_weight
```

- `min_hops(n, goal)` = shortest number of edges from n to goal (found via BFS on unweighted graph)
- `min_edge_weight` = smallest edge cost in the entire graph

### Heuristic Properties
* **Admissibility ($h(n) \le h^*(n)$):** To reach the goal, the robot *must* traverse at least $hops$ edges, each costing at least $d_{min}$. Therefore, $h(n)$ represents the absolute minimum possible cost and never overestimates the true cost.
* **Consistency ($h(n) \le c(n,m) + h(m)$):** The actual step cost between adjacent nodes $c(n,m)$ is always $\ge d_{min}$. This perfectly offsets the maximum possible 1-hop decrease in the remaining estimated cost.

With an admissible & consistent heuristic, IDA* is guaranteed to return the optimal path.

## 3. Cost Function

IDA* uses - Cost Function
Nodes are evaluated using: $$f(n) = g(n) + h(n)$$
* **$g(n)$**: Actual cumulative travel cost from the source to node $n$.
* **$h(n)$**: Estimated remaining cost from node $n$ to the destination.

The algorithm starts with threshold = h(source). Each iteration does a DFS but cuts off any path where f(n) exceeds the threshold. If we don't find the goal, the threshold gets bumped up to the smallest f-value that went over, and we try again. This way we gradually explore costlier paths until we hit the optimal one.

## 4. Why IDA*?

Reasons for choosing IDA* over other algorithms:

1. **Guaranteed optimal** — same as A* when heuristic is admissible
2. **Low memory** — only stores the current path (O(depth)), whereas A* keeps the whole open list in memory which can blow up
3. **Uses heuristic** — unlike BFS/Dijkstra which are blind, IDA* prunes bad paths early using h(n)
4. **Good fit for this problem** — the robot has limited battery so we want optimal routes without wasting computational resources


---

## 5. Implementation

In [113]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1 │ Bounded PathStack for DFS + Cycle Detection
# ─────────────────────────────────────────────────────────────────────────────
# PathStack maintains the current IDA* path with a fixed capacity equal to
# the number of nodes in the graph. It supports push/pop/peek, and uses an
# internal set for O(1) membership tests.
#   • Prevents stack overflow when path length would exceed graph size.
#   • Enables cycle avoidance in DFS by checking if a neighbor is already
#     in the current path.
#   • Provides path state for backtracking and visited-sequence reporting.
# ─────────────────────────────────────────────────────────────────────────────

from collections import deque
import math
import sys


class PathStack:
    """
    Stack data structure used to maintain the current DFS path in IDA*.
    Has a fixed capacity (= number of nodes in the graph) since the path
    can never be longer than visiting every node once.

    The stack methods print informative error messages when a push is
    attempted on a full stack or when pop/peek is attempted on an empty stack.
    """
    
    def __init__(self, capacity):
        self._stack = []
        self._capacity = capacity
        self._items_set = set()  # for O(1) membership checks (cycle detection)
    
    def push(self, item):
        """Add a node to the path. Prints error if stack is full."""
        if len(self._stack) >= self._capacity:
            print(f"Stack overflow: Cannot push '{item}' - path stack is full (capacity={self._capacity})")
            return False
        self._stack.append(item)
        self._items_set.add(item)
        return True
    
    def pop(self):
        """Remove and return the top node. Prints error if stack is empty."""
        if len(self._stack) == 0:
            print("Stack underflow: Cannot pop - path stack is empty!")
            return None
        item = self._stack.pop()
        self._items_set.discard(item)
        return item
    
    def peek(self):
        """Return top element without removing it."""
        if len(self._stack) == 0:
            print("Stack empty: Nothing to peek!")
            return None
        return self._stack[-1]
    
    def __contains__(self, item):
        """O(1) cycle detection."""
        return item in self._items_set
    
    def __len__(self):
        return len(self._stack)
    
    def to_list(self):
        """Return a copy of the stack as a list (bottom to top)."""
        return list(self._stack)
    
    def is_empty(self):
        return len(self._stack) == 0
    
    def is_full(self):
        return len(self._stack) >= self._capacity

### 5.1 Reading the Input File

In [114]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2 │ Input Parsing and Validation
# ─────────────────────────────────────────────────────────────────────────────
# read_input() reads a text file and converts it into structured test cases.
#   • Skips blank lines, looks for "Road Connections" blocks, then reads
#     edges until "Source" is encountered.
#   • Validates edge format, distinct endpoints, positive integer weights,
#     and non-empty source/destination values.
#   • Returns a list of cases where each case contains edges, source, and
#     destination.
# ─────────────────────────────────────────────────────────────────────────────


def read_input(filename):
    """Parse input file and return list of test cases."""
    
    # try opening the file
    try:
        with open(filename, 'r') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"Error: File '{filename}' does not exist!")
        return None
    except IOError as e:
        print(f"Error: Could not read file - {e}")
        return None
    
    # remove blank lines
    lines = [l.strip() for l in content.splitlines() if l.strip()]
    
    if len(lines) == 0:
        print("Error: Input file is empty!")
        return None
    
    cases = []
    i = 0
    
    while i < len(lines):
        # skip until we hit 'Road Connections'
        while i < len(lines) and 'road connections' not in lines[i].lower():
            i += 1
        if i >= len(lines):
            break
        i += 1  # move past that header line
        
        # now read the edges
        edges = []
        while i < len(lines) and not lines[i].lower().startswith('source'):
            line = lines[i]
            parts = line.split()
            
            if len(parts) == 3:
                # normal format like: MG_Road Electronic_City 2
                u, v = parts[0], parts[1]
                try:
                    w = int(parts[2])
                except ValueError:
                    print(f"Error: Weight '{parts[2]}' is not a valid integer in line: {line}")
                    return None
            else:
                print(
                    f"Error: Don't know how to parse this edge line: '{line}' - expected format 'U V W'"
                )
                return None
            
            if u == v:
                print(f"Error: Edge endpoints must be distinct, got '{u}-{v}'")
                return None
            
            # weight must be positive
            if w <= 0:
                print(f"Error: Edge weight must be positive, got {w} for {u}-{v}")
                return None
            
            edges.append((u, v, w))
            i += 1
        
        # grab the source
        if i >= len(lines) or 'source' not in lines[i].lower():
            print("Error: Expected 'Source:' line but didn't find it")
            return None
        source = lines[i].split(':', 1)[1].strip()
        if not source:
            print("Error: Source is blank!")
            return None
        i += 1
        
        # grab the destination
        if i >= len(lines) or 'destination' not in lines[i].lower():
            print("Error: Expected 'Destination:' line but didn't find it")
            return None
        destination = lines[i].split(':', 1)[1].strip()
        if not destination:
            print("Error: Destination is blank!")
            return None
        i += 1
        
        if len(edges) == 0:
            print("Error: No edges found for this case")
            return None
        
        cases.append({
            'edges': edges,
            'source': source,
            'destination': destination
        })
    
    if len(cases) == 0:
        print("Error: Couldn't find any valid test cases in the file")
        return None
    
    return cases

### 5.2 Building the Graph
Using an adjacency list (dictionary of lists). Since roads are bidirectional, we add each edge both ways.

In [115]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3 │ Graph Construction and Display
# ─────────────────────────────────────────────────────────────────────────────
# build_graph() constructs an undirected adjacency list from the edge list.
#   • Adds each edge in both directions because roads are bidirectional.
#   • Ensures every node appears in the graph dictionary even if it has
#     only one neighbor.
# print_graph() prints the adjacency list neatly.
# ─────────────────────────────────────────────────────────────────────────────


def build_graph(edges):
    """Create adjacency list from edge list. Undirected graph."""
    graph = {}
    
    for u, v, w in edges:
        if u not in graph:
            graph[u] = []
        if v not in graph:
            graph[v] = []
        
        # add both directions since roads are two-way
        graph[u].append((v, w))
        graph[v].append((u, w))
    
    if len(graph) == 0:
        print("Warning: Graph is empty, nothing was added")
    
    return graph


def print_graph(graph):
    """Display the graph nicely."""
    print("\nGraph (Adjacency List):")
    if len(graph) == 0:
        print("  <empty graph>")
        return
    for node in sorted(graph.keys()):
        connections = [f"{nbr}({w})" for nbr, w in graph[node]]
        print(f"  {node} --> {connections}")

### 5.3 Computing the Heuristic
We do a BFS from the goal backwards to find how many hops each node is away, then multiply by the cheapest edge in the graph.

In [116]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 │ Heuristic: BFS Hop Distance × Minimum Edge Cost
# ─────────────────────────────────────────────────────────────────────────────
# compute_heuristic() builds an admissible, consistent heuristic for IDA*.
#   • Finds the smallest edge weight in the entire graph: min_weight.
#   • Runs BFS from the goal node on the unweighted graph to compute the
#     minimum hop count from every node to the goal.
#   • Sets h(n) = hops(n, goal) × min_weight.
#   • If a node cannot reach the goal, h(n) = infinity.
# Because every actual move costs at least min_weight, this heuristic never
# overestimates and is consistent.
# ─────────────────────────────────────────────────────────────────────────────


def compute_heuristic(graph, goal):
    """Admissible heuristic: h(n) = min_hops_to_goal * min_edge_weight."""
    
    # first find the cheapest edge in the whole graph
    min_weight = math.inf
    for node in graph:
        for nbr, w in graph[node]:
            if w < min_weight:
                min_weight = w
    
    if min_weight == math.inf:
        # no edges at all
        print("Error: Graph has no edges!")
        return {}
    
    # BFS from goal to find hop distance to every reachable node
    hops = {goal: 0}
    queue = deque([goal])
    
    while queue:
        curr = queue.popleft()
        for neighbor, _ in graph[curr]:
            if neighbor not in hops:
                hops[neighbor] = hops[curr] + 1
                queue.append(neighbor)
    
    # now compute h(n) for each node
    h = {}
    for node in graph:
        if node in hops:
            h[node] = hops[node] * min_weight
        else:
            h[node] = math.inf  # can't reach goal from here
    
    return h

### 5.4 IDA* Algorithm
The main search. Does iterative deepening with f-cost as the bound.

In [117]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 5 │ IDA* Search with Iterative Thresholding
# ─────────────────────────────────────────────────────────────────────────────
# ida_star() performs the main search and returns optimal path metrics.
#   • Validates source/destination existence and handles source == goal.
#   • Computes the heuristic and sets initial threshold = h(source).
#   • Uses recursive DFS with f(n)=g(n)+h(n); prunes branches where
#     f > threshold.
#   • Tracks:
#       - nodes_evaluated: every node considered before pruning
#       - nodes_expanded: nodes that pass the threshold and generate children
#       - visited_per_iteration: nodes explored in each threshold iteration
#   • Avoids cycles by checking the current path set before pushing neighbors.
#   • When the goal is reached, returns the optimal path and cost.
#   • If no path exists, returns None and signals exhaustion.
# ─────────────────────────────────────────────────────────────────────────────


def ida_star(graph, source, destination):
    """
    IDA* search.
    Returns (path, cost, nodes_expanded, nodes_evaluated, visited_order)
    or (None, -1, 0, 0, []) if no path.
    """
    
    # check source exists
    if source not in graph:
        print(f"Error: Source '{source}' doesn't exist in the graph!")
        return None, -1, 0, 0, []
    
    # check destination exists
    if destination not in graph:
        print(f"Error: Destination '{destination}' doesn't exist in the graph!")
        return None, -1, 0, 0, []
    
    # trivial case
    if source == destination:
        return [source], 0, 1, 1, [[source]]
    
    # get heuristic values
    h = compute_heuristic(graph, destination)
    print(f"\n Heuristic values: {h}")
    
    if h.get(source, math.inf) == math.inf:
        print(f"Error: '{source}' cannot reach '{destination}' - no connecting path!")
        return None, -1, 0, 0, []
    
    # PathStack capacity = number of nodes (longest possible acyclic path)
    path = PathStack(capacity=len(graph))
    path.push(source)
    print(f" Initial threshold = h({source}) = {h[source]}")
    
    nodes_evaluated = [0]   # every node the algorithm looks at (including pruned)
    nodes_expanded = [0]    # nodes that pass threshold check and iterate neighbors
    visited_per_iteration = []   # list of lists, one per iteration
    current_iter_visited = []    # nodes that passed threshold in the current iteration
    answer = [None]
    iteration = [0]
    
    def dfs(g, threshold):
        """Recursive DFS with threshold cutoff."""
        node = path.peek()
        f = g + h[node]
        
        # count every node the algorithm evaluates (before threshold check)
        nodes_evaluated[0] += 1
        
        # if f exceeds threshold, cut this branch
        if f > threshold:
            print(f"   Pruned {node}: f({node}) = g({g}) + h({h[node]}) = {f} > threshold({threshold})")
            return f
        
        # node passed the threshold — record it in the visited sequence
        current_iter_visited.append(node)
        
        print(f"   Exploring {node}: g={g}, h={h[node]}, f={f} <= threshold({threshold})")
        print(f"   Current path: {' -> '.join(path.to_list())}")
        
        # did we reach the goal?
        if node == destination:
            answer[0] = (path.to_list(), g)
            print(f"   *** GOAL REACHED at {node} with cost {g} ***")
            print(f"   Final path: {' -> '.join(path.to_list())}")
            return -1  # signal that we found it
        
        # node is not the goal and passes threshold — it gets expanded
        nodes_expanded[0] += 1
        
        next_threshold = math.inf
        
        for neighbor, cost in graph[node]:
            # skip nodes already in current path (avoid cycles)
            if neighbor in path:
                print(f"   Skipping {neighbor} (already in path - cycle avoided)")
                continue
            
            path.push(neighbor)  # go deeper
            t = dfs(g + cost, threshold)
            
            if t == -1:
                return -1  # found! bubble up
            if t < next_threshold:
                next_threshold = t
            
            path.pop()  # backtrack
        
        return next_threshold
    
    # start with threshold = h(source)
    threshold = h[source]
    
    while True:
        iteration[0] += 1
        current_iter_visited.clear()
        print(f" --- Iteration {iteration[0]}, threshold = {threshold} ---")
        t = dfs(0, threshold)
        
        visited_per_iteration.append(list(current_iter_visited))
        
        if t == -1:
            # found the optimal path!
            print(f" Goal found in iteration {iteration[0]}!")
            return answer[0][0], answer[0][1], nodes_expanded[0], nodes_evaluated[0], visited_per_iteration
        
        if t == math.inf:
            # exhausted all possibilities, no path
            print("Search exhausted - no path to destination.")
            return None, -1, nodes_expanded[0], nodes_evaluated[0], visited_per_iteration
        
        # bump up threshold and try again
        print(f" Threshold raised: {threshold} -> {t}")
        # reset path to just the source for the next iteration
        while not path.is_empty():
            path.pop()
        path.push(source)
        threshold = t


### 5.5 Putting it all together

In [118]:

# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 │ Execution and Output Writing
# ─────────────────────────────────────────────────────────────────────────────
# solve() runs IDA* for each parsed test case and records results.
#   • Builds the graph, prints the graph, source, and destination.
#   • Runs ida_star() and prints the optimal path, cost, nodes expanded,
#     visited sequence per iteration, and nodes evaluated.
#   • Writes case summaries to an output file.
# ─────────────────────────────────────────────────────────────────────────────

def solve(cases, output_file='outputPSXX.txt'):
    """Run IDA* on all test cases and print + save results."""
    output_lines = []
    
    for idx, case in enumerate(cases, 1):
        print(f"\n{'='*55}")
        print(f"  CASE {idx}")
        print(f"{'='*55}")
        
        graph = build_graph(case['edges'])
        print_graph(graph)
        
        src = case['source']
        dst = case['destination']
        print(f"\nSource: {src}")
        print(f"Destination: {dst}")
        
        # run the search
        path, cost, expanded, evaluated, visited = ida_star(graph, src, dst)
        
        print(f"\n--- Results ---")
        if path is not None:
            path_str = ' -> '.join(path)
            
            # format visited sequence: show per iteration separated by |
            iter_strs = [' -> '.join(nodes) for nodes in visited]
            visited_str = ' | '.join(iter_strs)
            
            print(f"Optimal Path: {path_str}")
            print(f"Total Travel Cost: {cost} units")
            print(f"Nodes Explored: {expanded} nodes")
            print(f"\nVisited Sequence (per iteration, separated by |):")
            for i, nodes in enumerate(visited, 1):
                print(f"  Iteration {i}: {' -> '.join(nodes)}")
            print(f"\nNodes Evaluated: {evaluated} nodes")
            
            output_lines.append(f"Case {idx}:")
            output_lines.append(f"Source: {src}")
            output_lines.append(f"Destination: {dst}")
            output_lines.append(f"Optimal Path: {path_str}")
            output_lines.append(f"Total Travel Cost: {cost} units")
            output_lines.append(f"Nodes Explored: {expanded} nodes")
            output_lines.append(f"Visited Sequence (Pipe Separtion): {visited_str}")
            output_lines.append(f"Nodes Evaluated: {evaluated} nodes")
            output_lines.append('')
        else:
            print("No path found.")
            output_lines.append(f"Case {idx}: No path found")
            output_lines.append('')
    
    # save to file
    try:
        with open(output_file, 'w') as f:
            f.write('\n'.join(output_lines))
        print(f"\n\nResults saved to '{output_file}'")
    except IOError as e:
        print(f"Error: Could not write to output file - {e}")


### 5.6 Running on the Input File

The notebook example uses `inputPS01.txt` and writes to `outputPS01.txt`.
The script version now accepts both input and output arguments and also supports a regex pattern for the input file when it matches exactly one file name.

- Required input format: `node1 node2 weight`
- Compact tokens like `AB3` are not accepted
- `Nodes Explored` counts total nodes evaluated during search, including pruned nodes
- `Visited Sequence` records nodes evaluated during each IDA* iteration, including pruned nodes

In [119]:
# read from input file
INPUT_FILE = 'inputPS01.txt'
OUTPUT_FILE = 'outputPS01.txt'

cases = read_input(INPUT_FILE)

if cases is not None:
    print(f"Successfully loaded {len(cases)} case(s) from '{INPUT_FILE}'")
    solve(cases, OUTPUT_FILE)
else:
    print("Failed to load input. Please check the file and try again.")

Successfully loaded 2 case(s) from 'inputPS01.txt'

  CASE 1

Graph (Adjacency List):
  Electronic_City --> ['MG_Road(2)', 'Whitefield(2)']
  Hebbal --> ['Koramangala(3)', 'Yelahanka(5)']
  Jayanagar --> ['Whitefield(6)', 'Yelahanka(2)']
  Koramangala --> ['MG_Road(4)', 'Hebbal(3)']
  MG_Road --> ['Electronic_City(2)', 'Koramangala(4)']
  Whitefield --> ['Electronic_City(2)', 'Yelahanka(4)', 'Jayanagar(6)']
  Yelahanka --> ['Whitefield(4)', 'Jayanagar(2)', 'Hebbal(5)']

Source: MG_Road
Destination: Yelahanka

 Heuristic values: {'MG_Road': 6, 'Electronic_City': 4, 'Koramangala': 4, 'Whitefield': 2, 'Yelahanka': 0, 'Jayanagar': 2, 'Hebbal': 2}
 Initial threshold = h(MG_Road) = 6
 --- Iteration 1, threshold = 6 ---
   Exploring MG_Road: g=0, h=6, f=6 <= threshold(6)
   Current path: MG_Road
   Exploring Electronic_City: g=2, h=4, f=6 <= threshold(6)
   Current path: MG_Road -> Electronic_City
   Skipping MG_Road (already in path - cycle avoided)
   Exploring Whitefield: g=4, h=2, f=6 <= 

---

## 6. Error Handling Demonstrations

Below we test that our code handles edge cases gracefully.

In [120]:
print("--- Test: Invalid source node ---")
test_edges = [('X', 'Y', 5), ('Y', 'Z', 3)]
test_graph = build_graph(test_edges)
result = ida_star(test_graph, 'A', 'Z')  # 'A' is not in graph
print(f"Result: {result}")
print()

--- Test: Invalid source node ---
Error: Source 'A' doesn't exist in the graph!
Result: (None, -1, 0, 0, [])



In [121]:
print("--- Test: Invalid destination node ---")
test_edges = [('X', 'Y', 5), ('Y', 'Z', 3)]
test_graph = build_graph(test_edges)
result = ida_star(test_graph, 'X', 'W')  # 'W' not in graph
print(f"Result: {result}")
print()

--- Test: Invalid destination node ---
Error: Destination 'W' doesn't exist in the graph!
Result: (None, -1, 0, 0, [])



In [122]:
print("--- Test: Source == Destination ---")
test_edges = [('A', 'B', 2), ('B', 'C', 3)]
test_graph = build_graph(test_edges)
path, cost, expanded, evaluated, visited = ida_star(test_graph, 'A', 'A')
print(f"Path: {path}, Cost: {cost}, Expanded: {expanded}, Evaluated: {evaluated}")
print()


--- Test: Source == Destination ---
Path: ['A'], Cost: 0, Expanded: 1, Evaluated: 1



In [123]:
print("--- Test: No path between source and destination ---")
# Two separate components: {A,B} and {C,D}
test_edges = [('A', 'B', 1), ('C', 'D', 2)]
test_graph = build_graph(test_edges)
result = ida_star(test_graph, 'A', 'D')
print(f"Result: {result}")
print()

--- Test: No path between source and destination ---

 Heuristic values: {'A': inf, 'B': inf, 'C': 1, 'D': 0}
Error: 'A' cannot reach 'D' - no connecting path!
Result: (None, -1, 0, 0, [])



In [124]:
print("--- Test: Empty graph ---")
empty_graph = build_graph([])
result = ida_star(empty_graph, 'A', 'B')
print(f"Result: {result}")
print()

--- Test: Empty graph ---
Error: Source 'A' doesn't exist in the graph!
Result: (None, -1, 0, 0, [])



In [125]:
print("--- Test: Non-existent input file ---")
result = read_input('this_file_does_not_exist.txt')
print(f"Result: {result}")
print()

--- Test: Non-existent input file ---
Error: File 'this_file_does_not_exist.txt' does not exist!
Result: None



In [126]:
print("--- Test: Graph with only 2 nodes ---")
test_edges = [('Start', 'End', 10)]
test_graph = build_graph(test_edges)
path, cost, expanded, evaluated, visited = ida_star(test_graph, 'Start', 'End')
print(f"Path: {path}")
print(f"Cost: {cost}")
print(f"Expanded: {expanded}")
print(f"Evaluated: {evaluated}")
print()


--- Test: Graph with only 2 nodes ---

 Heuristic values: {'Start': 10, 'End': 0}
 Initial threshold = h(Start) = 10
 --- Iteration 1, threshold = 10 ---
   Exploring Start: g=0, h=10, f=10 <= threshold(10)
   Current path: Start
   Exploring End: g=10, h=0, f=10 <= threshold(10)
   Current path: Start -> End
   *** GOAL REACHED at End with cost 10 ***
   Final path: Start -> End
 Goal found in iteration 1!
Path: ['Start', 'End']
Cost: 10
Expanded: 1
Evaluated: 2

